# r3 · Rolling window, Elastic Net (B3.2)

Does the VIX gain depend on a long training sample that includes the GFC? We refit `en_no_macro` and `en_vix_only` on a 60-month rolling window instead of an expanding one, and repeat B3.2 (`en_vix_only` vs `en_no_macro`). Only these two models are rerun this way.

The forecasts are built at the end of `b2_elastic_net.ipynb` (models named `*_roll60`). The first forecast is still March 2008, since the burn-in is also 60 months.

Tests are the same as e3: Clark-West on log-RV MSE, DM on QLIKE, one-sided, Newey-West with the Andrews bandwidth.

Output: `models/forecasting/robustness/results/r3_rolling_evaluation.csv`, with the same columns as r1 but `window` instead of `sample`.


In [1]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import MODELS_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from research_config import BLOCKS, TICKERS

ROLLING_PATH = MODELS_DIR / "forecasting" / "robustness" / "forecasts" / "en_rolling60_forecasts.csv"
RESULTS_DIR = MODELS_DIR / "forecasting" / "robustness" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# B3.2: VIX on its own, on top of past volatility
pair_id, a, b, _ = next(p for p in BLOCKS["B3"]["pairs"] if p[0] == "B3.2")
SPECS = {"expanding": (a, b), "rolling 60": (f"{a}_roll60", f"{b}_roll60")}

expanding = pd.read_csv(FORECAST_DIR / "en_forecasts.csv", parse_dates=["date", "target_date"])
rolling = pd.read_csv(ROLLING_PATH, parse_dates=["date", "target_date"])
df = pd.concat([expanding, rolling], ignore_index=True)
missing = {m for pair in SPECS.values() for m in pair} - set(df["model"])
if missing:
    raise ValueError(f"Missing forecasts: {sorted(missing)}. Run the last section of b2 first.")
print(f"{pair_id}: {a} vs {b}")

B3.2: en_vix_only vs en_no_macro


In [2]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1


def one_sided_p(t):
    return float(1 - norm.cdf(t))

## Sample

Common sample: the (ETF, month) rows where all four forecasts (two expanding, two rolling) exist, so both windows are scored on the same months.


In [3]:
models = [m for pair in SPECS.values() for m in pair]
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]
if (d.groupby(["etf", "date"])["actual_logRV"].nunique() != 1).any():
    raise ValueError("Realized values differ across forecast files.")
d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"{d['target_date'].min():%Y-%m} to {d['target_date'].max():%Y-%m} (forecast months), {d['date'].nunique()} months")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
d.groupby("model")[LOSSES].mean().loc[models].round(4)

2008-03 to 2025-12 (forecast months), 214 months


,QLIKE,MSE_log,MSE_level
model,,,
en_vix_only,0.2953,0.4783,0.0013
en_no_macro,0.3279,0.5232,0.0001
en_vix_only_roll60,0.3002,0.5224,0.0118
en_no_macro_roll60,0.3796,0.5385,0.0001


## Tests


In [4]:
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
loss_idx = pd.concat([per_etf, pooled]).set_index(["model", "etf"])
n_months = d["date"].nunique()

rows = []
for window, (model_a, model_b) in SPECS.items():
    A = d[d["model"] == model_a].set_index(["date", "etf"])
    B = d[d["model"] == model_b].set_index(["date", "etf"])
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    # Clark-West adjusted differential; positive favours a
    cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)

    for etf in ["pooled"] + TICKERS:
        if etf == "pooled":
            dm_t, cw_t = dm_diff.groupby("date").mean(), cw.groupby("date").mean()
        else:
            dm_t, cw_t = dm_diff.xs(etf, level="etf"), cw.xs(etf, level="etf")
        for loss, test, t in [("QLIKE", "DM", -hac_t(dm_t)), ("MSE_log", "Clark-West", hac_t(cw_t))]:
            rows.append({
                "window": window, "pair": pair_id, "a": model_a, "b": model_b, "ETF": etf,
                "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(model_a, etf), loss] / loss_idx.loc[(model_b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t), "n_months": n_months,
            })

results = pd.DataFrame(rows)
results["pct_change"] = (results["loss_ratio"] - 1) * 100   # negative: a has the lower loss
results.to_csv(RESULTS_DIR / "r3_rolling_evaluation.csv", index=False)

# The expanding rows must reproduce the B3.2 rows of e3.
e3 = pd.read_csv(FORECAST_RESULTS_DIR / "e3_evaluation.csv").query("pair == @pair_id")
check = results[results["window"] == "expanding"].merge(
    e3, on=["pair", "ETF", "loss", "test"], suffixes=("", "_e3"))
assert len(check) == len(e3)
for col in ["loss_ratio", "gain_t", "p_value"]:
    assert np.allclose(check[col], check[col + "_e3"]), col
print("expanding rows match e3_evaluation.csv")

expanding rows match e3_evaluation.csv


## Pooled results

Each cell is `loss_ratio (p-value)`; a ratio below 1 means `en_vix_only` has the lower loss.


In [5]:
def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def cell(r, p):
    return f"{r:.3f} ({fmt_p(p)})"


pooled_tests = results[results["ETF"] == "pooled"]
mse = pooled_tests[pooled_tests["loss"] == "MSE_log"].set_index("window")
ql = pooled_tests[pooled_tests["loss"] == "QLIKE"].set_index("window")
pd.DataFrame({
    "Log MSE, ratio (Clark-West p)": [cell(r, p) for r, p in zip(mse["loss_ratio"], mse["p_value"])],
    "Log MSE, % change": [f"{v:+.2f}%" for v in mse["pct_change"]],
    "QLIKE, ratio (DM p)": [cell(r, p) for r, p in zip(ql["loss_ratio"], ql["p_value"])],
    "QLIKE, % change": [f"{v:+.2f}%" for v in ql["pct_change"]],
}, index=mse.index)

,"Log MSE, ratio (Clark-West p)","Log MSE, % change","QLIKE, ratio (DM p)","QLIKE, % change"
window,,,,
expanding,0.914 (<0.001),-8.59%,0.901 (0.002),-9.95%
rolling 60,0.970 (<0.001),-2.99%,0.791 (0.077),-20.92%


## By sector (exploratory)

Log-MSE loss ratio (Clark-West p-value) for `en_vix_only` vs `en_no_macro`, by sector ETF. One test per sector with no multiple-testing correction, so a single sector's p-value is weak evidence.


In [6]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}

t = results[results["loss"] == "MSE_log"].copy()
t["cell"] = [cell(r, p) for r, p in zip(t["loss_ratio"], t["p_value"])]
table = t.pivot(index="ETF", columns="window", values="cell").loc[TICKERS + ["pooled"], list(SPECS)]
table.index = [f"{e} {SECTORS[e]}" for e in TICKERS] + ["Pooled"]
table.rename_axis(None)

window,expanding,rolling 60
XLB Materials,0.886 (<0.001),0.996 (<0.001)
XLE Energy,0.990 (0.041),0.999 (0.023)
XLF Financials,0.848 (<0.001),0.842 (<0.001)
XLI Industrials,0.866 (<0.001),0.978 (<0.001)
XLY Cons. Disc.,1.009 (<0.001),1.067 (<0.001)
Pooled,0.914 (<0.001),0.970 (<0.001)
